# TP2: Image-to-Prompt Inversion with Metric-Guided Search

This notebook implements a single methodology for TP2:

**Metric-guided iterative prompt inversion**

The task is not image captioning. A prompt is considered good only if, when rendered again with the fixed `SimianLuo/LCM_Dreamshaper_v7` setup and the known seed, it produces an image visually close to the target.

The loop is:

`target image → candidate prompt → deterministic LCM render → image-side metrics → ranking → refinement`

The notebook includes:

- local target image loading;
- seed extraction from filenames;
- fixed LCM rendering;
- CLIP, LPIPS and MSE evaluation;
- candidate prompt expansion;
- batch evaluation and ranking;
- iterative prompt refinement;
- final prompt export.


## 1. Install Dependencies

Run this cell first in Colab. Ignore last errors.


In [ ]:
# Install the PyTorch Diffusers stack.
# Pillow 12 can break some Diffusers/Transformers imports in Colab Python 3.12
# with errors such as: cannot import name '_Ink' from PIL._typing.
#!pip install -q -U "diffusers[torch]" transformers accelerate safetensors matplotlib "pandas<3"
#!pip install -q -U --force-reinstall "Pillow<12"
#!pip install -q -U lpips

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
torch.cuda.empty_cache()
print(torch.cuda.memory_summary())

In [ ]:
import os
import re
import gc
import csv
import json
import time
import zipfile
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass

import numpy as np
import pandas as pd

from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display

from torchvision import transforms
from diffusers import DiffusionPipeline, LCMScheduler
from transformers import CLIPModel, CLIPProcessor

import lpips

CACHE_DIR = Path.cwd() / "model_cache"
CACHE_DIR.mkdir(exist_ok=True)

os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(CACHE_DIR)

torch.hub.set_dir(str(CACHE_DIR / "torch_hub"))
gc.collect()
torch.cuda.empty_cache()

print("Imports loaded successfully.")
print("Cache directory:", CACHE_DIR)

## 2. Local Paths and GPU Setup

This notebook uses local files and GPU acceleration.

In [ ]:
# Local project paths (no Google Drive)
WORKSPACE_ROOT = Path.cwd()
OUTPUT_DIR = WORKSPACE_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

def list_target_images(path):
    path = Path(path)
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
        return [path]
    if not path.exists():
        return []
    return sorted(p for p in path.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)

# Local target directories
TARGET_DIR_CANDIDATES = [
    WORKSPACE_ROOT / "tp2-chosen",
    WORKSPACE_ROOT / "students" / "tp2-chosen",
    WORKSPACE_ROOT / "tp2_targets",
]

ZIP_CANDIDATES = [
    WORKSPACE_ROOT / "tp2-chosen.zip",
    WORKSPACE_ROOT / "students" / "tp2-chosen.zip",
]

# Extract first available zip if no folder with images exists yet.
if not any(list_target_images(candidate) for candidate in TARGET_DIR_CANDIDATES):
    for zip_path in ZIP_CANDIDATES:
        if zip_path.exists():
            extract_dir = WORKSPACE_ROOT / "tp2-chosen"
            extract_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(extract_dir)
            print(f"Extracted {zip_path} -> {extract_dir}")
            break

TARGET_DIR = None
for candidate in TARGET_DIR_CANDIDATES:
    if list_target_images(candidate):
        TARGET_DIR = candidate
        break

if TARGET_DIR is None:
    raise FileNotFoundError(
        "No target images found. Put images in tp2-chosen/ directory, "
        "or place tp2-chosen.zip in the workspace root to auto-extract."
    )

target_images = list_target_images(TARGET_DIR)
print("Target folder:", TARGET_DIR)
print("Output folder:", OUTPUT_DIR)
print("Number of targets:", len(target_images))
target_images

## 3. Utilities: Seeds, Display, and Output Folders

The target filename encodes the render seed:

- `7836.png` uses seed `7836`;
- `1159_25.png` uses seed `1159`.


In [ ]:
def seed_from_filename(path, fallback=2026):
    match = re.match(r"^(\d+)", Path(path).stem)
    return int(match.group(1)) if match else fallback


def safe_stem(path):
    return "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in Path(path).stem)


def load_image(path):
    return Image.open(path).convert("RGB")


def create_run_dir(base_dir=OUTPUT_DIR, identity="student_run"):
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    run_dir = Path(base_dir) / f"{timestamp}_{identity}"
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir

def write_csv(path, rows):
    rows = list(rows)
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        Path(path).write_text("")
        return
    fieldnames = []
    for row in rows:
        for key in row.keys():
            if key not in fieldnames:
                fieldnames.append(key)
    with open(path, "w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def show_images(paths, cols=3, title=None):
    paths = list(paths)
    if not paths:
        print("No images to show.")
        return
    rows = (len(paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    if rows == 1 and cols == 1:
        axes = [[axes]]
    elif rows == 1:
        axes = [axes]
    elif cols == 1:
        axes = [[ax] for ax in axes]
    for ax in [ax for row in axes for ax in row]:
        ax.axis("off")
    for ax, path in zip([ax for row in axes for ax in row], paths):
        ax.imshow(load_image(path))
        ax.set_title(Path(path).name)
        ax.axis("off")
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()


for path in target_images:
    print(Path(path).name, "-> seed", seed_from_filename(path))


In [ ]:
show_images(target_images, cols=3, title="TP2 target images")


## 4. Load the LCM (Latent Consistency Model) Generator with GPU Support

These are the fixed generation settings used for TP2 targets. GPU acceleration is automatically enabled if CUDA is available.

In [ ]:
@dataclass(frozen=True)
class LCMConfig:
    model_id: str = "SimianLuo/LCM_Dreamshaper_v7"
    num_inference_steps: int = 8
    guidance_scale: float = 8.0
    lcm_origin_steps: int = 50
    width: int = 512
    height: int = 512

config = LCMConfig()


def default_device():
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


device = default_device()
print("Using device:", device)


def load_lcm_pipeline(config):
    dtype = torch.float16 if device == "cuda" else torch.float32
    pipe = DiffusionPipeline.from_pretrained(
        config.model_id,
        torch_dtype=dtype,
        use_safetensors=True,
        cache_dir=CACHE_DIR
    )
    if hasattr(pipe, "safety_checker"):
        pipe.safety_checker = None
    pipe.scheduler = LCMScheduler.from_config(pipe.scheduler.config)
    pipe.enable_attention_slicing()
    pipe.enable_vae_slicing()
    pipe.to(device)
    return pipe


pipe = load_lcm_pipeline(config)


## 5. Generate and Save Images

Use `render_prompt_for_target(...)` to render a prompt with the seed encoded in the target filename.

In [ ]:
def render_prompt(prompt, seed, pipe=pipe, config=config):
    generator_device = "cpu" if device == "mps" else device
    generator = torch.Generator(device=generator_device).manual_seed(seed) # fixa seed para cada renderização com base no nome do arquivo de destino
    image = pipe(
        prompt=prompt,
        num_inference_steps=config.num_inference_steps,
        guidance_scale=config.guidance_scale,
        lcm_origin_steps=config.lcm_origin_steps,
        width=config.width,
        height=config.height,
        output_type="pil",
        generator=generator,
    ).images[0]
    return image


def render_prompt_for_target(prompt, target_path):
    seed = seed_from_filename(target_path)
    return render_prompt(prompt, seed=seed)


def save_generated_image(image, run_dir, target_path, prompt_index=1, prompt=None):
    """
    Save generated image to disk.
    If prompt is provided, includes it in metadata file.
    """
    target_dir = Path(run_dir) / safe_stem(target_path)
    target_dir.mkdir(parents=True, exist_ok=True)
    path = target_dir / f"candidate_{prompt_index:03d}.png"
    image.save(path)
    
    # Optional: save prompt metadata alongside image
    if prompt is not None:
        meta_path = target_dir / f"candidate_{prompt_index:03d}_prompt.txt"
        meta_path.write_text(prompt)
    
    return path

## 6. Metrics: Evaluating Prompt Quality

Without metrics, we cannot objectively measure if a prompt is good or bad. This section implements three complementary metrics:

- **CLIP Similarity**: Semantic similarity between target and generated image (0-1, higher is better)
- **LPIPS Distance**: Perceptual distance based on deep features (0+, lower is better)  
- **Pixel MSE**: Direct pixel-level difference (0+, lower is better)

In [ ]:
METRIC_DEVICE = "cpu"
clip_model_id = "openai/clip-vit-base-patch32"

gc.collect()
torch.cuda.empty_cache()

clip_model = CLIPModel.from_pretrained(
    clip_model_id,
    cache_dir=CACHE_DIR
).to(METRIC_DEVICE).eval()

clip_processor = CLIPProcessor.from_pretrained(
    clip_model_id,
    cache_dir=CACHE_DIR
)

torch.hub.set_dir(str(CACHE_DIR / "torch_hub"))

lpips_model = lpips.LPIPS(net="alex").to(METRIC_DEVICE).eval()

print("Metrics loaded on:", METRIC_DEVICE)
print("CLIP model:", clip_model_id)

In [ ]:
def clip_image_similarity(img_a, img_b):
    inputs = clip_processor(
        images=[img_a, img_b],
        return_tensors="pt"
    ).to(METRIC_DEVICE)

    with torch.no_grad():
        outputs = clip_model.vision_model(pixel_values=inputs["pixel_values"])
        features = clip_model.visual_projection(outputs.pooler_output)
        features = features / features.norm(dim=-1, keepdim=True)

    return torch.matmul(features[0], features[1]).item()


def image_to_lpips_tensor(img):
    """Convert PIL image to normalized tensor for LPIPS evaluation."""
    transform = transforms.Compose([
        transforms.Resize((config.width, config.height)),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])
    return transform(img).unsqueeze(0).to(METRIC_DEVICE)


def lpips_distance(img_a, img_b):
    """
    Compute LPIPS perceptual distance.
    Lower values indicate more similar images.
    """
    tensor_a = image_to_lpips_tensor(img_a)
    tensor_b = image_to_lpips_tensor(img_b)

    with torch.no_grad():
        distance = lpips_model(tensor_a, tensor_b).item()

    return distance


def pixel_mse(img_a, img_b):
    """
    Compute pixel-level Mean Squared Error.
    Lower values indicate more similar images.
    """
    img_a = img_a.resize((config.width, config.height))
    img_b = img_b.resize((config.width, config.height))

    arr_a = np.asarray(img_a).astype(np.float32) / 255.0
    arr_b = np.asarray(img_b).astype(np.float32) / 255.0

    return np.mean((arr_a - arr_b) ** 2)


def evaluate_image_pair(target_img, generated_img):
    """
    Evaluate similarity between target and generated image.
    Returns a dictionary with all three metrics.
    """
    return {
        "clip_similarity": clip_image_similarity(target_img, generated_img),
        "lpips": lpips_distance(target_img, generated_img),
        "mse": pixel_mse(target_img, generated_img),
    }

## 7. Methodology: Metric-Guided Iterative Prompt Inversion

We formulate the problem as black-box optimization over text prompts.

For each target image, we search for a prompt `p` that minimizes the visual distance between the target and the image generated by the fixed LCM configuration.

Because the checkpoint, scheduler, inference steps, guidance scale, resolution and seed are fixed, each prompt maps deterministically to a generated image. Therefore the search space is text, but the evaluation happens on images.

The practical strategy is:

1. manually describe each target image using visual attributes;
2. expand each description into candidate prompts using controlled prompt templates;
3. render every candidate with the fixed seed of the corresponding target;
4. evaluate each generated image using CLIP similarity, LPIPS and MSE;
5. rank prompts using a combined score;
6. refine the top prompts through small prompt mutations;
7. select the best final prompt per target.


In [ ]:
# Prompt expansion utilities

def unique_preserve_order(items):
    seen = set()
    unique_items = []
    for item in items:
        item = " ".join(str(item).strip().split())
        if item and item not in seen:
            seen.add(item)
            unique_items.append(item)
    return unique_items


def join_prompt_parts(*parts):
    """Join prompt fragments while removing empty parts."""
    return ", ".join(
        part.strip(" ,")
        for part in parts
        if isinstance(part, str) and part.strip()
    )


STYLE_MODIFIERS = [
    "realistic photograph",
    "highly detailed realistic image",
    "professional photography",
    "cinematic realistic photograph",
]

LIGHTING_MODIFIERS = [
    "soft studio lighting",
    "warm directional lighting",
    "soft shadows",
    "natural soft light",
]

CAMERA_MODIFIERS = [
    "close-up",
    "shallow depth of field",
    "sharp focus on the main subject",
    "slightly blurred background",
]

QUALITY_MODIFIERS = [
    "high detail",
    "clean composition",
    "balanced colors",
]


def expand_prompt_from_analysis(analysis, max_prompts=40):
    """
    Build candidate prompts from a structured visual analysis.

    Required key:
    - subject: the main visual content of the target image

    Optional keys:
    - composition, background, style, lighting, camera, details, negative
    """
    subject = analysis.get("subject", "").strip()
    if not subject or "TODO" in subject:
        raise ValueError(
            "Each target needs a non-empty 'subject'. "
            "Fill TARGET_ANALYSIS before running the evaluation."
        )

    composition = analysis.get("composition", "")
    background = analysis.get("background", "")
    details = analysis.get("details", "")

    user_styles = analysis.get("style", [])
    user_lighting = analysis.get("lighting", [])
    user_camera = analysis.get("camera", [])

    if isinstance(user_styles, str):
        user_styles = [user_styles]
    if isinstance(user_lighting, str):
        user_lighting = [user_lighting]
    if isinstance(user_camera, str):
        user_camera = [user_camera]

    styles = unique_preserve_order(user_styles + STYLE_MODIFIERS)
    lightings = unique_preserve_order(user_lighting + LIGHTING_MODIFIERS)
    cameras = unique_preserve_order(user_camera + CAMERA_MODIFIERS)

    candidates = []

    # Short baseline prompts
    candidates.append(subject)
    candidates.append(join_prompt_parts(subject, composition))
    candidates.append(join_prompt_parts(subject, composition, background))

    # Controlled combinations
    for style in styles:
        for lighting in lightings[:3]:
            for camera in cameras[:3]:
                prompt = join_prompt_parts(
                    subject,
                    composition,
                    style,
                    lighting,
                    camera,
                    background,
                    details,
                    "high detail"
                )
                candidates.append(prompt)

    # More compact variants often work better than overloaded prompts
    for style in styles[:3]:
        candidates.append(join_prompt_parts(subject, style, composition, lighting if (lighting := lightings[0]) else "", background))
        candidates.append(join_prompt_parts(subject, style, cameras[0], background, details))

    return unique_preserve_order(candidates)[:max_prompts]


## 8. Manual Visual Analysis for Each Target

Fill this dictionary before running the evaluation.

This is the only manual part of the methodology. The goal is not to write the final prompt directly, but to describe the visual ingredients that the search process will expand and test.

Recommended fields:

- `subject`: main object/person/scene;
- `composition`: framing, object placement, viewpoint;
- `background`: setting, environment, blur, color;
- `style`: visual style candidates;
- `lighting`: light direction, warmth, shadows;
- `camera`: close-up, wide shot, shallow depth of field, etc.;
- `details`: secondary objects, textures, colors, important elements.


In [ ]:
# Fill this cell manually after looking at the target images above.
# Keep the keys generated from the filenames. Replace the TODO text for each target.

TARGET_ANALYSIS = {
    safe_stem(path): {
        "subject": "TODO: describe the main subject of this target image",
        "composition": "TODO: describe framing, viewpoint and object placement",
        "background": "TODO: describe the background or environment",
        "style": [
            "realistic photograph",
        ],
        "lighting": [
            "soft studio lighting",
        ],
        "camera": [
            "close-up",
            "shallow depth of field",
        ],
        "details": "TODO: add important colors, textures and secondary objects",
    }
    for path in target_images
}

TARGET_ANALYSIS


In [ ]:
def build_candidate_bank(target_analysis, max_prompts_per_target=40):
    """
    Convert TARGET_ANALYSIS into a dictionary:
    {
        target_stem: [prompt_1, prompt_2, ...]
    }
    """
    candidate_bank = {}
    for target_path in target_images:
        key = safe_stem(target_path)
        if key not in target_analysis:
            raise KeyError(f"Missing analysis for target: {key}")
        candidate_bank[key] = expand_prompt_from_analysis(
            target_analysis[key],
            max_prompts=max_prompts_per_target
        )
    return candidate_bank


def preview_candidate_bank(candidate_bank, n=5):
    for key, prompts in candidate_bank.items():
        print(f"\n--- {key}: {len(prompts)} prompts ---")
        for prompt in prompts[:n]:
            print("-", prompt)


# After filling TARGET_ANALYSIS, uncomment:
# candidate_bank = build_candidate_bank(TARGET_ANALYSIS, max_prompts_per_target=40)
# preview_candidate_bank(candidate_bank, n=5)


## 9. Batch Render, Evaluation and Ranking

This section renders each candidate prompt with the fixed seed of its target image and computes the image-side metrics.

The combined score is computed per target, not globally, so each target is ranked against its own candidate prompts.

Score convention:

- higher CLIP similarity is better;
- lower LPIPS is better;
- lower MSE is better;
- higher final `score` is better.


In [ ]:
def evaluate_prompt_for_target(prompt, target_path, run_dir, prompt_index, iteration=0, family="initial"):
    target_img = load_image(target_path)
    generated_img = render_prompt_for_target(prompt, target_path)
    metrics = evaluate_image_pair(target_img, generated_img)

    output_path = save_generated_image(
        generated_img,
        run_dir=run_dir,
        target_path=target_path,
        prompt_index=prompt_index,
        prompt=prompt
    )

    row = {
        "target": Path(target_path).name,
        "target_key": safe_stem(target_path),
        "seed": seed_from_filename(target_path),
        "iteration": int(iteration),
        "family": family,
        "prompt_index": int(prompt_index),
        "prompt": prompt,
        "generated_path": str(output_path),
        "clip_similarity": float(metrics["clip_similarity"]),
        "lpips": float(metrics["lpips"]),
        "mse": float(metrics["mse"]),
    }

    del generated_img
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row


def add_combined_score(df, clip_weight=0.50, lpips_weight=0.40, mse_weight=0.10):
    """
    Normalize metrics per target and compute one combined score.

    Higher score = better prompt.
    """
    df = df.copy()

    def normalize_group(group):
        group = group.copy()

        def minmax(series):
            return (series - series.min()) / (series.max() - series.min() + 1e-8)

        group["clip_norm"] = minmax(group["clip_similarity"])
        group["lpips_norm"] = minmax(group["lpips"])
        group["mse_norm"] = minmax(group["mse"])

        group["score"] = (
            clip_weight * group["clip_norm"]
            + lpips_weight * (1.0 - group["lpips_norm"])
            + mse_weight * (1.0 - group["mse_norm"])
        )
        return group

    return df.groupby("target", group_keys=False).apply(normalize_group)


def prompts_for_target(candidate_prompts, target_path):
    """
    Accept either:
    - a list of prompts shared by all targets;
    - a dict mapping target_key or target filename to prompt lists.
    """
    if isinstance(candidate_prompts, list):
        return candidate_prompts

    target_key = safe_stem(target_path)
    target_name = Path(target_path).name

    if target_key in candidate_prompts:
        return candidate_prompts[target_key]
    if target_name in candidate_prompts:
        return candidate_prompts[target_name]

    raise KeyError(f"No prompts found for target {target_name} / {target_key}")


def run_evaluation(candidate_prompts, target_images, identity="prompt_inversion", iteration=0, family="initial"):
    """
    Render and evaluate all candidate prompts for all target images.
    Saves:
    - generated images;
    - raw_results.csv/json;
    - ranked_results.csv;
    - top3_results.csv.
    """
    run_dir = create_run_dir(identity=identity)
    rows = []

    for target_path in target_images:
        target_prompt_list = prompts_for_target(candidate_prompts, target_path)
        print(f"\nEvaluating {Path(target_path).name}: {len(target_prompt_list)} prompts")

        for prompt_index, prompt in enumerate(target_prompt_list, start=1):
            print(f"  [{prompt_index:03d}/{len(target_prompt_list):03d}] {prompt[:90]}")
            row = evaluate_prompt_for_target(
                prompt=prompt,
                target_path=target_path,
                run_dir=run_dir,
                prompt_index=prompt_index,
                iteration=iteration,
                family=family
            )
            rows.append(row)

    raw_df = pd.DataFrame(rows)
    ranked_df = add_combined_score(raw_df)
    ranked_df = ranked_df.sort_values(
        ["target", "score", "clip_similarity"],
        ascending=[True, False, False]
    ).reset_index(drop=True)

    top3_df = ranked_df.groupby("target", group_keys=False).head(3).reset_index(drop=True)

    write_csv(run_dir / "raw_results.csv", rows)
    ranked_df.to_csv(run_dir / "ranked_results.csv", index=False)
    top3_df.to_csv(run_dir / "top3_results.csv", index=False)

    # Safe JSON export: all values are already plain Python strings/floats/ints.
    (run_dir / "raw_results.json").write_text(
        json.dumps(rows, indent=2, ensure_ascii=False)
    )

    print("\nSaved results to:", run_dir)
    return ranked_df, top3_df, run_dir


## 10. First Evaluation Round

After filling `TARGET_ANALYSIS`, run this cell.

It creates an initial prompt bank, renders the candidates, ranks them, and shows the top 3 prompts per target.


In [ ]:
# 1) Build initial candidates from the manual visual analysis
candidate_bank = build_candidate_bank(TARGET_ANALYSIS, max_prompts_per_target=40)
preview_candidate_bank(candidate_bank, n=3)

# 2) Render + evaluate + rank
results_df, top3_df, run_dir = run_evaluation(
    candidate_bank,
    target_images,
    identity="round0_initial_prompt_expansion",
    iteration=0,
    family="initial_expansion"
)

display(top3_df[[
    "target", "prompt", "clip_similarity", "lpips", "mse", "score", "generated_path"
]])


## 11. Visualise the Best Candidates

This view is important because metrics are useful but imperfect. Use it to identify which visual attributes are missing, excessive, or misplaced before the refinement round.


In [ ]:
def show_topk(top_df, k=3):
    for target_name in top_df["target"].unique():
        target_path = next(p for p in target_images if Path(p).name == target_name)
        target_img = load_image(target_path)
        rows = top_df[top_df["target"] == target_name].head(k).reset_index(drop=True)

        fig, axes = plt.subplots(1, k + 1, figsize=(4 * (k + 1), 4))

        axes[0].imshow(target_img)
        axes[0].set_title(f"Target\n{target_name}")
        axes[0].axis("off")

        for i, row in rows.iterrows():
            img = load_image(row["generated_path"])
            axes[i + 1].imshow(img)
            axes[i + 1].set_title(
                f"#{i+1} | Score {row['score']:.3f}\n"
                f"CLIP {row['clip_similarity']:.3f} | LPIPS {row['lpips']:.3f} | MSE {row['mse']:.4f}"
            )
            axes[i + 1].axis("off")

        plt.tight_layout()
        plt.show()


show_topk(top3_df, k=3)


## 12. Refinement Round

The refinement round mutates the best prompts from the previous evaluation.

This keeps the methodology simple and explainable:

- keep the strongest candidates;
- add or replace visual modifiers;
- test the new prompts with the same deterministic generator;
- rank again with the same metrics.

Do not create too many refinement rounds unless the metrics or visual inspection clearly improve.


In [ ]:
REFINEMENT_SUFFIXES = [
    "more accurate composition",
    "closer to the reference image",
    "stronger focus on the main subject",
    "cleaner background",
    "more realistic textures",
    "more faithful colors",
    "balanced lighting",
    "detailed foreground",
]

REFINEMENT_REPLACEMENTS = [
    ("realistic photograph", "ultra realistic photograph"),
    ("soft studio lighting", "warm soft studio lighting"),
    ("close-up", "tight close-up"),
    ("shallow depth of field", "very shallow depth of field"),
    ("slightly blurred background", "softly blurred background"),
]


def mutate_prompt(prompt):
    mutations = [prompt]

    # Add controlled suffixes
    for suffix in REFINEMENT_SUFFIXES:
        mutations.append(join_prompt_parts(prompt, suffix))

    # Replace common modifiers
    for old, new in REFINEMENT_REPLACEMENTS:
        if old in prompt:
            mutations.append(prompt.replace(old, new))

    # Compact version: sometimes shorter prompts work better
    parts = [part.strip() for part in prompt.split(",") if part.strip()]
    if len(parts) > 5:
        mutations.append(", ".join(parts[:5]))

    return unique_preserve_order(mutations)


def build_refined_candidate_bank(top_df, top_k=3, max_prompts_per_target=30):
    refined_bank = {}

    for target_name in top_df["target"].unique():
        rows = top_df[top_df["target"] == target_name].head(top_k)
        target_key = safe_stem(target_name)

        prompts = []
        for prompt in rows["prompt"].tolist():
            prompts.extend(mutate_prompt(prompt))

        refined_bank[target_key] = unique_preserve_order(prompts)[:max_prompts_per_target]

    return refined_bank


refined_bank = build_refined_candidate_bank(top3_df, top_k=3, max_prompts_per_target=30)
preview_candidate_bank(refined_bank, n=5)


In [ ]:
refined_results_df, refined_top3_df, refined_run_dir = run_evaluation(
    refined_bank,
    target_images,
    identity="round1_metric_guided_refinement",
    iteration=1,
    family="metric_guided_refinement"
)

display(refined_top3_df[[
    "target", "prompt", "clip_similarity", "lpips", "mse", "score", "generated_path"
]])

show_topk(refined_top3_df, k=3)


## 13. Final Selection and Export

This cell compares the best prompts from the initial and refinement rounds, selects the best prompt per target, and exports the final table.

The exported `final_prompts.csv` is the main deliverable.


In [ ]:
all_results_df = pd.concat([results_df, refined_results_df], ignore_index=True)
all_results_df = add_combined_score(all_results_df)

final_df = (
    all_results_df
    .sort_values(["target", "score", "clip_similarity"], ascending=[True, False, False])
    .groupby("target", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

final_export_path = OUTPUT_DIR / "final_prompts.csv"
final_df.to_csv(final_export_path, index=False)

display(final_df[[
    "target", "seed", "prompt", "clip_similarity", "lpips", "mse", "score", "generated_path"
]])

print("Final prompts exported to:", final_export_path)


## 14. Methodology Summary for the Report

Use this text as a concise explanation of the implemented method.

> We formulate image-to-prompt inversion as a black-box optimization problem over text prompts. Since the diffusion generator, seed, scheduler, resolution, inference steps and guidance scale are fixed, each prompt deterministically maps to a generated image. The goal is therefore to find the prompt whose rendered image minimizes a visual distance to the target. Our methodology starts from a structured visual analysis of each target image, expands it with controlled prompt modifiers related to subject, composition, lighting, camera and style, renders each candidate with the fixed LCM configuration, and ranks the outputs using a combined image-side metric based on CLIP similarity, LPIPS and MSE. The best candidates are iteratively refined through prompt mutations, and the final prompt is selected according to the best metric score.
